In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('use_inf_as_na', True)

In [ ]:
liq_data = pd.read_csv('bid_ask.csv')
liq_data

In [ ]:
rolling_five = []
for j in liq_data.TICKER.unique():
    for i in range(len(liq_data[liq_data.TICKER == j])):
        rolling_five.append(liq_data[i:i+5].agg({'BIDLO': 'min','ASKHI': 'max','VOL': 'sum','SHROUT': 'mean','PRC': 'mean'}))


In [ ]:
rolling_five_df = pd.DataFrame(rolling_five)
rolling_five_df.columns = ['bidlo_min', 'askhi_max', 'vol_sum','shrout_mean', 'prc_mean']
liq_vol_all = pd.concat([liq_data,rolling_five_df], axis=1)
tickers = liq_vol_all.TICKER.unique()
liq_vol_all

In [ ]:
liq_ratio = []
Lhh = []
turnover_ratio = []

for j in tickers:
    for i in range(len(liq_vol_all[liq_vol_all.TICKER == j])):
        liq_ratio.append((liq_vol_all['PRC'][i+1:i+6] *liq_vol_all['VOL'][i+1:i+6]).sum()/(np.abs(liq_vol_all['PRC'][i+1:i+6].mean() -liq_vol_all['PRC'][i:i+5].mean())))
        Lhh.append((liq_vol_all['PRC'][i:i+5].max() -liq_vol_all['PRC'][i:i+5].min()) /liq_vol_all['PRC'][i:i+5].min() /(liq_vol_all['VOL'][i:i+5].sum() /liq_vol_all['SHROUT'][i:i+5].mean() *liq_vol_all['PRC'][i:i+5].mean()))
        turnover_ratio.append((1/liq_vol_all['VOL'].count()) *(np.sum(liq_vol_all['VOL'][i:i+1]) /np.sum(liq_vol_all['SHROUT'][i:i+1])))

In [ ]:
liq_vol_all['liq_ratio'] = pd.DataFrame(liq_ratio)
liq_vol_all['Lhh'] = pd.DataFrame(Lhh)
liq_vol_all['turnover_ratio'] = pd.DataFrame(turnover_ratio)

In [ ]:
liq_vol_all['mid_price'] = (liq_vol_all.ASKHI + liq_vol_all.BIDLO) / 2
liq_vol_all['percent_quoted_ba'] = (liq_vol_all.ASKHI -liq_vol_all.BIDLO) / liq_vol_all.mid_price
liq_vol_all['percent_effective_ba'] = 2 * abs((liq_vol_all.PRC -liq_vol_all.mid_price)) / liq_vol_all.mid_price
liq_vol_all

In [ ]:
liq_vol_all['price_diff'] = liq_vol_all.groupby('TICKER')['PRC'].apply(lambda x:x.diff()).reset_index(level=0, drop=True)
liq_vol_all.dropna(inplace=True)

roll = []

for j in tickers:
    for i in range(len(liq_vol_all[liq_vol_all.TICKER == j])):
        roll_cov = np.cov(liq_vol_all['price_diff'][i:i+5],liq_vol_all['price_diff'][i+1:i+6])
        if roll_cov[0,1] < 0:
            roll.append(2 * np.sqrt(-roll_cov[0, 1]))
        else:
            roll.append(2 * np.sqrt(np.abs(roll_cov[0, 1])))

In [ ]:
gamma = []
beta = []

for j in tickers:
    for i in range(len(liq_vol_all[liq_vol_all.TICKER == j])):
        gamma.append((max(liq_vol_all['ASKHI'].iloc[i+1],liq_vol_all['ASKHI'].iloc[i]) -min(liq_vol_all['BIDLO'].iloc[i+1],liq_vol_all['BIDLO'].iloc[i])) ** 2)
        beta.append((liq_vol_all['ASKHI'].iloc[i+1] -liq_vol_all['BIDLO'].iloc[i+1]) ** 2 +(liq_vol_all['ASKHI'].iloc[i] -liq_vol_all['BIDLO'].iloc[i]) ** 2)

beta_array = np.array(beta)
gamma_array = np.array(gamma)

In [ ]:
liq_vol_all['vol_diff_pct'] = liq_vol_all.groupby('TICKER')['VOL'].apply(lambda x: x.diff()).reset_index(level=0, drop=True)
liq_vol_all['price_diff_pct'] = liq_vol_all.groupby('TICKER')['PRC'].apply(lambda x: x.diff()).reset_index(level=0, drop=True)
liq_vol_all

In [ ]:
alpha = ((np.sqrt(2 * beta_array) - np.sqrt(beta_array)) /(3 - (2 * np.sqrt(2)))) - np.sqrt(gamma_array /(3 - (2 * np.sqrt(2))))
CS_spread = (2 * np.exp(alpha - 1)) / (1 + np.exp(alpha))

In [ ]:
liq_vol_all = liq_vol_all.reset_index()
liq_vol_all['roll'] = pd.DataFrame(roll)
liq_vol_all['CS_spread'] = pd.DataFrame(CS_spread)

In [ ]:
dvol = []
amihud = []
florackis = []
cet = []

for j in tickers:
    for i in range(len(liq_vol_all[liq_vol_all.TICKER == j])):
        dvol.append((liq_vol_all['PRC'][i:i+5] *liq_vol_all['VOL'][i:i+5]).sum())

liq_vol_all['dvol'] = pd.DataFrame(dvol)

In [ ]:
for j in tickers:
    for i in range(len(liq_vol_all[liq_vol_all.TICKER == j])):
        amihud.append((1 / liq_vol_all['RET'].count()) *(np.sum(np.abs(liq_vol_all['RET'][i:i+1])) /np.sum(liq_vol_all['dvol'][i:i+1])))
        florackis.append((1 / liq_vol_all['RET'].count()) *(np.sum(np.abs(liq_vol_all['RET'][i:i+1]) /liq_vol_all['turnover_ratio'][i:i+1])))
        cet.append(np.sum(liq_vol_all['vol_diff_pct'][i:i+1])/np.sum(liq_vol_all['price_diff_pct'][i:i+1]))

liq_vol_all['amihud'] = pd.DataFrame(amihud)
liq_vol_all['florackis'] = pd.DataFrame(florackis)
liq_vol_all['cet'] = pd.DataFrame(cet)

In [ ]:
import statsmodels.api as sm
liq_vol_all['VOL_pct_change'] = liq_vol_all.groupby('TICKER')['VOL'].apply(lambda x: x.pct_change()).reset_index(level=0, drop=True)
liq_vol_all.dropna(subset=['VOL_pct_change'], inplace=True)

In [ ]:
unsys_resid = []

for i in tickers:
    X1 = liq_vol_all[liq_vol_all['TICKER'] == i]['vwretx']
    y = liq_vol_all[liq_vol_all['TICKER'] == i]['RET']
    ols = sm.OLS(y, X1).fit()
    unsys_resid.append(ols.resid)

In [ ]:
market_impact = {}

for i, j in zip(liq_vol_all.TICKER.unique(),range(len(liq_vol_all['TICKER'].unique()))):
    X2 = liq_vol_all[liq_vol_all['TICKER'] == i]['VOL_pct_change']
    ols = sm.OLS(unsys_resid[j] ** 2, X2).fit()
    market_impact[j] = ols.resid

liq_vol_all['market_impact'] = pd.concat([market_impact[0], market_impact[1], market_impact[2]], axis="rows")

In [ ]:
cols = ['vol_diff_pct', 'price_diff_pct', 'price_diff','VOL_pct_change', 'dvol', 'mid_price']
liq_measures_all = liq_vol_all.drop(liq_vol_all[cols], axis=1).iloc[:, -11:]
liq_measures_all.dropna(inplace=True)
liq_measures_all.describe().T

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

In [ ]:
liq_measures_all2 = liq_measures_all.dropna()
scaled_liq = StandardScaler().fit_transform(liq_measures_all2)
kwargs = dict(alpha=0.5, bins=50, stacked=True)
plt.hist(liq_measures_all['percent_quoted_ba'],**kwargs, label='TC-based')
plt.hist(liq_measures_all['turnover_ratio'],**kwargs, label='Volume-based')
plt.hist(liq_measures_all['market_impact'],**kwargs, label='Market-based')
plt.title('Multi Modality of the Liquidity Measures')
plt.legend()
plt.show()

In [ ]:
def cluster_state(data, nstates):
    gmm = GaussianMixture(n_components=nstates,covariance_type='spherical',init_params='kmeans')
    gmm_fit = gmm.fit(scaled_liq)
    #labels = gmm_fit.predict(scaled_liq)
    state_probs = gmm.predict_proba(scaled_liq)
    state_probs_df = pd.DataFrame(state_probs,columns=['state-1','state-2','state-3'])
    state_prob_means = [state_probs_df.iloc[:, i].mean()for i in range(len(state_probs_df.columns))]
    pos = np.argmax(state_prob_means)
    print('State-{:d} is likely to occur with a probability of {:4f}'.format(pos+1,state_prob_means[pos]))
    return state_probs

In [ ]:
state_probs = cluster_state(scaled_liq, 3)
print(f'State probabilities are {state_probs.mean(axis=0)}')

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
def gmm_pca(data, nstate):
    pca = PCA(n_components=3)
    components = pca.fit_transform(data)
    mxtd = GaussianMixture(n_components=nstate,covariance_type='spherical')
    gmm = mxtd.fit(components)
    labels = gmm.predict(components)
    state_probs = gmm.predict_proba(components)
    return state_probs,pca

In [ ]:
state_probs, pca = gmm_pca(scaled_liq, 3)
print(f'State probabilities are {state_probs.mean(axis=0)}')

In [ ]:
def wpc():
    state_probs_df = pd.DataFrame(state_probs,columns=['state-1', 'state-2','state-3'])
    state_prob_means = [state_probs_df.iloc[:, i].mean()for i in range(len(state_probs_df.columns))]
    pos = np.argmax(state_prob_means)
    print('State-{:d} is likely to occur with a probability of {:4f}'.format(pos+1,state_prob_means[pos]))

In [ ]:
wpc()

In [ ]:
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loading_matrix = pd.DataFrame(loadings,columns=['PC1', 'PC2', 'PC3'],index=liq_measures_all.columns)
loading_matrix